# Capstone：已授权的云端一键完整实验

此入口用于当前用户已授权的独立云端实验。打开后选择 **运行时 → 全部运行**；如 Colab 提示，允许本 notebook 访问现有 Secret **Capstone**。代码、数据和模型请求实际在 Google 托管 Colab 执行。不要连接本地运行时，不要在个人电脑执行代码。

**执行范围**：完整 800 道 MemoryAgentBench + 589 道 PersonaMem，共 4,167 个逻辑组别结果。固定 DeepSeek 模型与既有 provider，不换模型；最多 4 个并发线程、每轮每分片 60 次请求、最多 12 个分片 × 10 轮，即本次调用最多 7,200 次尝试。每个 canonical 请求在冻结 journal 中最多 2 次尝试。实际用量和失败全部保留；未核实价格，不虚构金额成本。

**点击全部运行会启动已授权的付费 API 调用**，不再要求修改开关。此文件本身无执行输出，尚无新效果结果。原有历史负结果、运行目录和协议保持不变。新工作目录为 `/content/cbt-v3.1-authorized-runall-20261105`。

**保留边界**：每轮结束后在 Colab 内生成白名单审核 ZIP；不会自动下载到你的 Mac，也不会自动挂载 Drive。`/content` 属于临时运行时，**运行时被删除时，这些文件也可能消失**。因此这不是跨运行时持久保存方案。结束后保留当前 Colab 运行时，等待云端控制恢复后转存受控云端工件；若需要 Drive 等持久保存，须另行明确配置。不要把完整基准文本或 Secret 提交 GitHub。

本实验只能评价公开任务的记忆回答表现。MAB 按一个相关 family 处理，PersonaMem 仅 20 个 connected groups；Persona 的 raw/managed 仍是原话控制组。不能由这些分数宣称 CBT 疗效或真人收益。


In [ ]:
import os,sys,json,hashlib,subprocess,importlib.util
from pathlib import Path
if not (Path('/content').is_dir() and os.environ.get('COLAB_RELEASE_TAG') and importlib.util.find_spec('google.colab') is not None):
    raise RuntimeError('Genuine hosted Colab required. Do not connect a local runtime or spoof this guard.')
CODE='1511706b3a4b0ac71416998b5b4d752a13c4e313'
REPO=Path('/content/cbt-objective-v3.1-authorized-'+CODE[:12])
WORK=Path('/content/cbt-v3.1-authorized-runall-20261105')
WORK.mkdir(exist_ok=True)
DATA=WORK/'registered-data'
SUITE=WORK/'full-800-589'
CHECKPOINTS=WORK/'checkpoints'
CHECKPOINTS.mkdir(exist_ok=True)
try:
    if not (REPO/'.git').exists():
        subprocess.run(['git','clone','--quiet','--no-checkout','https://github.com/nanlalala/cbt-memory-system.git',str(REPO)],check=True,capture_output=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--quiet','--detach',CODE],check=True,capture_output=True)
    if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True,stderr=subprocess.PIPE).strip()!=CODE:
        raise RuntimeError('Executable revision mismatch')
    if subprocess.run(['git','-C',str(REPO),'diff','--quiet','HEAD'],capture_output=True).returncode:
        raise RuntimeError('The pinned checkout has modifications')
except Exception as error:
    raise RuntimeError('Pinned cloud checkout unavailable; no provider call started. Error type: '+type(error).__name__) from None
print(json.dumps({'execution':'genuine hosted Google Colab','executable':CODE,'work_directory':str(WORK),'ephemeral_runtime_storage':True,'new_model_calls_at_this_step':0}))


## 恢复真实旧实验的曝光身份

下面只恢复原有 protocol 和 gold-free metadata 的原始字节，不含答案或完整历史，不伪造新协议。


In [ ]:
# Exact gold-free original pilot metadata and protocol, recovered from the
# retained checksum-verified audit. Original protocol/bytes stay unchanged.
EXPOSURE_DIR=WORK/'development-source-v1'
EXPOSURE_DIR.mkdir(exist_ok=True)
EXPOSURE=EXPOSURE_DIR/'cases.jsonl'
EXPOSURE_PROTOCOL=EXPOSURE_DIR/'protocol.json'
EXPOSURE_TEXT='{"case_id": "mab:factconsolidation_mh_262k_no82", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_mh_262k", "query": "Where was the work location of the individual who founded the religion practiced by Edward Mooney?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_mh_262k:9c8ce1ed911d25a3", "source_question_index": 82, "source_context_id": "factconsolidation_mh_262k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "58c564c5918cbd403c8325b4eb3fa813c30276f16be453acb0805f7b5074cd75"}\n{"case_id": "mab:factconsolidation_mh_262k_no9", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_mh_262k", "query": "What is the name of the person who created the language used as the official language of the headquarters of FC Viikingit?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_mh_262k:9c8ce1ed911d25a3", "source_question_index": 9, "source_context_id": "factconsolidation_mh_262k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "58c564c5918cbd403c8325b4eb3fa813c30276f16be453acb0805f7b5074cd75"}\n{"case_id": "mab:factconsolidation_mh_32k_no93", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_mh_32k", "query": "In which language does the government of the country of citizenship of W. G. Sebald conduct official business?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_mh_32k:2a44d6ad3264a775", "source_question_index": 93, "source_context_id": "factconsolidation_mh_32k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "2e54864d8c5fe82f264ee388c0f6f3328bc629649947ca4348785e9b8014b03b"}\n{"case_id": "mab:factconsolidation_mh_32k_no33", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_mh_32k", "query": "Which city serves as the capital of the country where the sport played by Brook Lopez originates?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_mh_32k:2a44d6ad3264a775", "source_question_index": 33, "source_context_id": "factconsolidation_mh_32k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "2e54864d8c5fe82f264ee388c0f6f3328bc629649947ca4348785e9b8014b03b"}\n{"case_id": "mab:factconsolidation_mh_64k_no58", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_mh_64k", "query": "From which country did the sport, in which Mel Daniels participates, originate?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_mh_64k:e2789821b46e6626", "source_question_index": 58, "source_context_id": "factconsolidation_mh_64k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "0bfd3ec543fa5354c172e027ac00b8112352f422d1f2221e6811290e60e3a730"}\n{"case_id": "mab:factconsolidation_mh_64k_no87", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_mh_64k", "query": "From which university did the partner of Harald V of Norway receive their education?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_mh_64k:e2789821b46e6626", "source_question_index": 87, "source_context_id": "factconsolidation_mh_64k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "0bfd3ec543fa5354c172e027ac00b8112352f422d1f2221e6811290e60e3a730"}\n{"case_id": "mab:factconsolidation_mh_6k_no8", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_mh_6k", "query": "To which continent does the country of citizenship of Colin Irwin pertain?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_mh_6k:e46cb14b53eedd71", "source_question_index": 8, "source_context_id": "factconsolidation_mh_6k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "25ab1ec2c44ba888f7055acdde531a0ea8a080532a23588d7a7bf16ea6dc0728"}\n{"case_id": "mab:factconsolidation_mh_6k_no99", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_mh_6k", "query": "In what language are the official documents of the country where Alta IF originates written?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_mh_6k:e46cb14b53eedd71", "source_question_index": 99, "source_context_id": "factconsolidation_mh_6k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "25ab1ec2c44ba888f7055acdde531a0ea8a080532a23588d7a7bf16ea6dc0728"}\n{"case_id": "mab:factconsolidation_sh_262k_no86", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_sh_262k", "query": "What is Alexander Hamilton famous for?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_sh_262k:9c8ce1ed911d25a3", "source_question_index": 86, "source_context_id": "factconsolidation_sh_262k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "7fc10f04ddcf23da177c2a16996aab9294e61082c1e7c8bdd5d27a270abd0566"}\n{"case_id": "mab:factconsolidation_sh_262k_no10", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_sh_262k", "query": "What is the country of citizenship of Cy Young?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_sh_262k:9c8ce1ed911d25a3", "source_question_index": 10, "source_context_id": "factconsolidation_sh_262k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "7fc10f04ddcf23da177c2a16996aab9294e61082c1e7c8bdd5d27a270abd0566"}\n{"case_id": "mab:factconsolidation_sh_32k_no69", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_sh_32k", "query": "Who is the author of The Marriage of Figaro?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_sh_32k:2a44d6ad3264a775", "source_question_index": 69, "source_context_id": "factconsolidation_sh_32k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "78e01a433ae6177fc1c4d5abd36c2017c6e82356759c039a929becc629a5b475"}\n{"case_id": "mab:factconsolidation_sh_32k_no38", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_sh_32k", "query": "What position does Lisa Leslie play?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_sh_32k:2a44d6ad3264a775", "source_question_index": 38, "source_context_id": "factconsolidation_sh_32k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "78e01a433ae6177fc1c4d5abd36c2017c6e82356759c039a929becc629a5b475"}\n{"case_id": "mab:factconsolidation_sh_64k_no51", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_sh_64k", "query": "Who performed Taxman?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_sh_64k:e2789821b46e6626", "source_question_index": 51, "source_context_id": "factconsolidation_sh_64k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "65e4ff9a52b8e9406d73cfe9088592c40ed71101b883e7bfaf4b768d9f911bc7"}\n{"case_id": "mab:factconsolidation_sh_64k_no14", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_sh_64k", "query": "Who is the director of CERN?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_sh_64k:e2789821b46e6626", "source_question_index": 14, "source_context_id": "factconsolidation_sh_64k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "65e4ff9a52b8e9406d73cfe9088592c40ed71101b883e7bfaf4b768d9f911bc7"}\n{"case_id": "mab:factconsolidation_sh_6k_no62", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_sh_6k", "query": "Who is the chief executive officer of Twitter?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_sh_6k:e46cb14b53eedd71", "source_question_index": 62, "source_context_id": "factconsolidation_sh_6k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "228122db06454385e331d766899b10ed52e758623aa3ef9c23a96a41bd94279b"}\n{"case_id": "mab:factconsolidation_sh_6k_no80", "group_id": "memoryagentbench:factconsolidation-family", "subject_id": "memoryagentbench:factconsolidation-family", "dataset_key": "memoryagentbench_conflict", "task": "factconsolidation_sh_6k", "query": "What position does Lisa Leslie play?", "options": [], "events": [], "as_of": null, "ordering_kind": "source_order_no_calendar", "history_scope_id": "mab:factconsolidation_sh_6k:e46cb14b53eedd71", "source_question_index": 80, "source_context_id": "factconsolidation_sh_6k", "clustering_note": "all variants conservatively grouped; exploratory family", "history_inventory_sha256": "228122db06454385e331d766899b10ed52e758623aa3ef9c23a96a41bd94279b"}\n'
EXPOSURE_SHA='d9cd969e9f9d22d3f9b35626433d6fd809e9ad3c4cf60616ddfd89651e6b29e9'
EXPOSURE_PROTOCOL_TEXT='{\n  "api_base": "https://tokendance.space/gateway/v1",\n  "arms": [\n    "short_term",\n    "raw_history_rag",\n    "managed_memory_rag"\n  ],\n  "budget_policy": "same character budgets and generation max_tokens; input token counts observed, not exact-token matched",\n  "case_ids": [\n    "mab:factconsolidation_mh_262k_no82",\n    "mab:factconsolidation_mh_262k_no9",\n    "mab:factconsolidation_mh_32k_no93",\n    "mab:factconsolidation_mh_32k_no33",\n    "mab:factconsolidation_mh_64k_no58",\n    "mab:factconsolidation_mh_64k_no87",\n    "mab:factconsolidation_mh_6k_no8",\n    "mab:factconsolidation_mh_6k_no99",\n    "mab:factconsolidation_sh_262k_no86",\n    "mab:factconsolidation_sh_262k_no10",\n    "mab:factconsolidation_sh_32k_no69",\n    "mab:factconsolidation_sh_32k_no38",\n    "mab:factconsolidation_sh_64k_no51",\n    "mab:factconsolidation_sh_64k_no14",\n    "mab:factconsolidation_sh_6k_no62",\n    "mab:factconsolidation_sh_6k_no80"\n  ],\n  "code_sha256": {\n    "evaluation/end-to-end-dialogue/execution_lock.py": "4a3da7b74236c09ed6e34afe781473e2f7276972f1e23b4410397ee2784b10d3",\n    "evaluation/end-to-end-dialogue/run_three_arm.py": "74a9b957fb9bbb4f19018e3b1ed27c03bd9544eeafb3861aa20d957537ac6a9e",\n    "evaluation/memory-retrieval/benchmark_adapters.py": "5457e5e45f710bafcae16679ac7c5df7b3fe17cf8e73d3cd2f1850c0e3168743",\n    "evaluation/memory-retrieval/benchmark_manifest.json": "b0e213c000f71bd0650e6c0a6648cac1cd62c774f9bdf8f646885ee481388a52",\n    "evaluation/memory-retrieval/validate_external_benchmarks.py": "70358b38a2fa872f0daed6adda3b8d294b883525becc04ab60264ce6753f0ab0",\n    "evaluation/objective-memory/dataset_registry.json": "28735fe2cb3b79a0b6a7c564e3f9df95d22594c05f87e713f816dd22006d15ca",\n    "evaluation/objective-memory/objective_datasets.py": "1eb3f095b96c2e05b554b5ec3baa76c5a9217d811e444a77e0dbce3ffb647c66",\n    "evaluation/objective-memory/run_objective_comparison.py": "735d3e6f0cac2774fb2255725ad38f89d87f545634a7fe4fa1a04dd8bff8925b",\n    "evaluation/objective-memory/score_objective_comparison.py": "262243267a1b2f73bf1838d0c88f50afc235402c9d934252982a38a0e3ffb719",\n    "modules/long-term-memory/lifecycle/memory_manager.py": "f9d29bc76c32a8a738fce872edd3de6415e127f978f4e7ea193ced137c48a183",\n    "modules/long-term-memory/retrieval/memory_retriever.py": "851f1cb74d9a2c74c0b8fe38d18f7179269b841dc69b15837f0eebf24aba86db",\n    "modules/long-term-memory/retrieval/memory_use_policy.py": "6a16e1572ca44a21c0daf227ef8d61ad241e67ec4717ef78cf09d12364254a65",\n    "modules/long-term-memory/schema/memory_models.py": "cd48ab95d153051ee9c2876f3dd300a7ffda9c1fb8a2d2708f1136c79f1064b9",\n    "modules/long-term-memory/storage/sqlite_store.py": "7af2316eb2a954d12be685fede4da8e90b9c83eda2db9b9a495d55f2b96fe16b",\n    "modules/model_snapshots.json": "ff2a9419bca405f247a493d2859c9906af6ce32c4007950305743b5c02bef039",\n    "modules/model_snapshots.py": "0b4f7a9bc46629ab12ad98552cc21f6a236b3e6c4eb923afdbd077f82ff9edf8"\n  },\n  "cohort": {\n    "case_count": 16,\n    "case_ids": [\n      "mab:factconsolidation_mh_262k_no82",\n      "mab:factconsolidation_mh_262k_no9",\n      "mab:factconsolidation_mh_32k_no93",\n      "mab:factconsolidation_mh_32k_no33",\n      "mab:factconsolidation_mh_64k_no58",\n      "mab:factconsolidation_mh_64k_no87",\n      "mab:factconsolidation_mh_6k_no8",\n      "mab:factconsolidation_mh_6k_no99",\n      "mab:factconsolidation_sh_262k_no86",\n      "mab:factconsolidation_sh_262k_no10",\n      "mab:factconsolidation_sh_32k_no69",\n      "mab:factconsolidation_sh_32k_no38",\n      "mab:factconsolidation_sh_64k_no51",\n      "mab:factconsolidation_sh_64k_no14",\n      "mab:factconsolidation_sh_6k_no62",\n      "mab:factconsolidation_sh_6k_no80"\n    ],\n    "group_count": 1,\n    "group_ids": [\n      "memoryagentbench:factconsolidation-family"\n    ]\n  },\n  "conditions": [\n    "short_term",\n    "raw_history_rag",\n    "managed_memory_rag"\n  ],\n  "dataset_manifest": {\n    "registered_inventory": {\n      "calendar_timestamps_invented": false,\n      "claim": "inventory only; no outcome, power or independence claim",\n      "distinct_visible_prefixes": 8,\n      "generation_cases_sha256": "72aa6bcad63153e77b806ab8efaa851e47794d601c261591cd9524857571b42a",\n      "independent_clusters": 1,\n      "input_files": {\n        "data": {\n          "bytes": 1491588,\n          "name": "Conflict_Resolution-00000-of-00001.parquet",\n          "sha256": "24d5c3f09ce0ce15625cb9f8a98f44f0d864ca6c94d7b4ad04eb697ca3a5ff45"\n        }\n      },\n      "maximum_visible_turns": 18333,\n      "minimum_visible_turns": 456,\n      "personas": 0,\n      "questions": 800,\n      "questions_are_independent_users": false,\n      "questions_per_cluster": {\n        "memoryagentbench:factconsolidation-family": 800\n      },\n      "scoring_references_sha256": "55e62a48de598671534cfd49c5f24ec88637721d6275590fd8c3e14d84f83208",\n      "source_contexts": 8,\n      "tasks": {\n        "factconsolidation_mh_262k": 100,\n        "factconsolidation_mh_32k": 100,\n        "factconsolidation_mh_64k": 100,\n        "factconsolidation_mh_6k": 100,\n        "factconsolidation_sh_262k": 100,\n        "factconsolidation_sh_32k": 100,\n        "factconsolidation_sh_64k": 100,\n        "factconsolidation_sh_6k": 100\n      }\n    },\n    "sampling": "two per released task sorted by SHA256(case_id), fixed before generation; exploratory pilot only; not full 800-question result",\n    "selected_case_ids": [\n      "mab:factconsolidation_mh_262k_no82",\n      "mab:factconsolidation_mh_262k_no9",\n      "mab:factconsolidation_mh_32k_no93",\n      "mab:factconsolidation_mh_32k_no33",\n      "mab:factconsolidation_mh_64k_no58",\n      "mab:factconsolidation_mh_64k_no87",\n      "mab:factconsolidation_mh_6k_no8",\n      "mab:factconsolidation_mh_6k_no99",\n      "mab:factconsolidation_sh_262k_no86",\n      "mab:factconsolidation_sh_262k_no10",\n      "mab:factconsolidation_sh_32k_no69",\n      "mab:factconsolidation_sh_32k_no38",\n      "mab:factconsolidation_sh_64k_no51",\n      "mab:factconsolidation_sh_64k_no14",\n      "mab:factconsolidation_sh_6k_no62",\n      "mab:factconsolidation_sh_6k_no80"\n    ]\n  },\n  "deduplication": {\n    "arm_requests": 48,\n    "case_count": 16,\n    "interpretation": "equivalent quote controls; shared response is not an independent observation",\n    "policy": "exact_messages_hash_deduplicated_call",\n    "raw_managed_equal_cases": 16,\n    "raw_managed_equal_fraction": 1.0,\n    "unique_provider_requests": 32\n  },\n  "evaluation_policy": {\n    "bootstrap_samples": 10000,\n    "bootstrap_seed": 20261104,\n    "independence_unit": "group_id",\n    "power_status": "exploratory_no_prospective_power",\n    "primary": "group_equal_accuracy"\n  },\n  "evaluator_identity": {\n    "metric_version": "released-upstream-substring-exact-match-or-option-parser-v1",\n    "path": "evaluation/objective-memory/score_objective_comparison.py",\n    "sha256": "262243267a1b2f73bf1838d0c88f50afc235402c9d934252982a38a0e3ffb719"\n  },\n  "file_sha256": {\n    "cases.jsonl": "d9cd969e9f9d22d3f9b35626433d6fd809e9ad3c4cf60616ddfd89651e6b29e9",\n    "inventory.jsonl": "093c12f4e806f88b076f2b1b0e70cdf4b73c48537cd21819a4a1a9adac374af5",\n    "prompts.jsonl": "99e405a1829a965944602d83ee032a6fac15b6743ed1708a74ce24fbfe018455",\n    "references.jsonl": "2c737fcd0da5d49593cc2c505032d99acf5dbcb6329a742dac6be6b8e243d9dc"\n  },\n  "fixture_only": false,\n  "knowledge_policy": "empty in every arm; this is memory QA, not CBT clinical outcome",\n  "maximum_attempts_per_request": 2,\n  "model_identity_policy": {\n    "accepted_provider_models": [\n      "deepseek-v4.1-flash",\n      "deepseek-v4-1-flash-260910"\n    ],\n    "policy_version": "reported-name-allowlist-v1",\n    "requested_model": "deepseek-v4.1-flash",\n    "verification_boundary": "preapproved reported-name matching only; not cryptographic backend identity"\n  },\n  "protocol_sha256": "9b798544eaf0f7b4355eb7b2772e199b45f80420dc648154942db54f0751d4fd",\n  "protocol_version": "objective-public-qa-quote-control-v1",\n  "representation": {\n    "clock_policy": "released timestamps if present; otherwise internal source-order schema clock never rendered as calendar",\n    "current_use": false,\n    "events_policy": "no inferred events; explicit lifecycle events use separate replay",\n    "origin_policy": "all released data system_observation, including synthetic role=user; unconfirmed candidates, no user consent fabricated",\n    "semantic_candidate_artifact": null,\n    "source_roles": "all released visible roles, retained equally in raw and managed",\n    "version": "source_verified_quote_v1"\n  },\n  "retrieval_model_identity": null,\n  "settings": {\n    "accepted_provider_models": [\n      "deepseek-v4-1-flash-260910"\n    ],\n    "backend": "lexical",\n    "dense_model": "intfloat/multilingual-e5-small",\n    "long_chars": 12000,\n    "max_context_chars": 24000,\n    "max_response_tokens": 256,\n    "minimum_relevance": 0.05,\n    "model": "deepseek-v4.1-flash",\n    "seed": 20261014,\n    "short_chars": 6000,\n    "short_turns": 16,\n    "thinking_mode": "disabled",\n    "top_k": 12\n  },\n  "source_text_policy": "common boundary-whitespace trim and whitespace-only exclusion; inventory retains original source bytes",\n  "temperature": 0\n}\n'
EXPOSURE_PROTOCOL_SHA='bb71df0ba73f053d7a7fa0b839af51b7803d3a768b47429fdda3fc068af049d2'
for path,text,expected in ((EXPOSURE,EXPOSURE_TEXT,EXPOSURE_SHA),(EXPOSURE_PROTOCOL,EXPOSURE_PROTOCOL_TEXT,EXPOSURE_PROTOCOL_SHA)):
    payload=text.encode('utf-8')
    if hashlib.sha256(payload).hexdigest()!=expected:
        raise RuntimeError('Embedded development-source bytes differ; no provider call started.')
    if path.exists():
        if hashlib.sha256(path.read_bytes()).hexdigest()!=expected:
            raise RuntimeError('Existing development-source bytes differ; preserve them and use a new run directory.')
    else:
        path.write_bytes(payload)
print('Exact original pilot protocol and gold-free cases restored; no answers or source-history text included.')


## 冻结协议、前台运行、逐轮审核检查点

所有模型调用前先冻结完整分母与参数；等待每轮全部工作结束再导出。遇到 inflight 或错误停止，不自动冒险补发。只有全部计划结果均已终结才计算完整套件分数。


In [ ]:
# Run All is the launch action for the user's already authorized finite batch.
# No paid toggle edit is needed. This cell stays in the foreground until it ends.
import fcntl,zipfile,tempfile,time
from datetime import datetime,timezone
def assert_pinned_module_origins(repo,loaded_modules,tracked_paths=None):
    # Fail before Secret access/preparation if this kernel still holds code
    # from a different checkout. Never silently unload a running module.
    if tracked_paths is None:
        tracked_paths=subprocess.check_output(['git','-C',str(repo),'ls-tree','-r','--name-only',CODE],text=True,stderr=subprocess.PIPE).splitlines()
    expected={}
    for relative in tracked_paths:
        if not relative.endswith('.py'):continue
        path=repo/relative
        expected.setdefault(path.stem,set()).add(path.resolve())
    for name,module in list(loaded_modules.items()):
        local_name=name.rsplit('.',1)[-1]
        if local_name not in expected:continue
        origin=getattr(module,'__file__',None)
        if not origin or Path(origin).resolve() not in expected[local_name]:
            raise RuntimeError('A repository module is cached from another executable. Restart the hosted Python session (keep runtime files), then Run All. No provider call started.')

assert_pinned_module_origins(REPO,sys.modules)
sys.path.insert(0,str(REPO/'evaluation/objective-memory'))
import v3_suite_driver as suite_driver
import v3_objective_audit as audit
assert_pinned_module_origins(REPO,sys.modules)
from google.colab import userdata
MAX_PASSES=10
WORKERS=4
CALLS_PER_SHARD_PER_PASS=60
LAUNCH_POLICY={
    'launcher_id':'authorized-colab-runall-v1','executable':CODE,
    'suite_version':suite_driver.VERSION,'planned_cases':1389,'planned_arm_rows':4167,
    'maximum_passes_per_invocation':MAX_PASSES,'maximum_workers':WORKERS,
    'maximum_calls_per_shard_per_pass':CALLS_PER_SHARD_PER_PASS,
    'maximum_shards':12,'maximum_attempts_per_invocation':7200,
    'maximum_attempts_per_canonical_request':2,
    'model':suite_driver.MODEL,'accepted_provider_model':suite_driver.ACCEPTED_MODEL,
    'api_base':suite_driver.API_BASE,'exposure_cases_sha256':EXPOSURE_SHA,
    'exposure_protocol_file_sha256':EXPOSURE_PROTOCOL_SHA,
    'work_directory':str(WORK),'old_runs_overwritten':False,
    'storage':'ephemeral Colab runtime; archives do not survive runtime deletion without a separately approved cloud transfer',
    'automatic_download_or_drive_mount':False,'clinical_claim':False,
}
policy_bytes=(json.dumps(LAUNCH_POLICY,sort_keys=True,ensure_ascii=False,indent=2)+'\n').encode()
policy_path=WORK/'launcher_manifest.json'
if policy_path.exists() and policy_path.read_bytes()!=policy_bytes:
    raise RuntimeError('Existing launcher policy differs; preserve it and choose a new run root.')
if not policy_path.exists():policy_path.write_bytes(policy_bytes)
print(json.dumps({'authorized_finite_launch_policy':LAUNCH_POLICY,'policy_sha256':hashlib.sha256(policy_bytes).hexdigest()}))


def checkpoint_pass_locked(suite, label):
    """Workers have stopped before this runs; only the existing audited allowlist is exported."""
    plan=json.loads((SUITE/'suite_plan.json').read_text())
    if len(plan['shards'])!=12:raise RuntimeError('Unexpected shard count')
    folder=CHECKPOINTS/label
    folder.mkdir(exist_ok=False)
    for shard in plan['shards']:
        audit.export_audit_zip(SUITE/'shards'/shard['shard_id'],folder/(shard['shard_id']+'.zip'))
    # Suite orchestration source is bound separately from per-shard reader code.
    driver_path=REPO/'evaluation/objective-memory/v3_suite_driver.py'
    driver_bytes=driver_path.read_bytes()
    expected=plan['code_sha256']['evaluation/objective-memory/v3_suite_driver.py']
    if hashlib.sha256(driver_bytes).hexdigest()!=expected:
        raise RuntimeError('Suite driver differs from its frozen plan')
    members={p.name:p.read_bytes() for p in folder.glob('*.zip')}
    for name in ('suite_plan.json','preparation_summary.json','execution_summary.json','execution_progress.json','suite_summary.json','suite_scores.jsonl','suite_progress.json'):
        path=SUITE/name
        if path.exists():
            if path.is_symlink() or not path.is_file():raise RuntimeError('Unsafe suite artifact')
            members['suite/'+name]=path.read_bytes()
    members['code/v3_suite_driver.py']=driver_bytes
    members['launcher_manifest.json']=policy_bytes
    sums={name:hashlib.sha256(value).hexdigest() for name,value in members.items()}
    members['SHA256SUMS.json']=(json.dumps(sums,sort_keys=True,indent=2)+'\n').encode()
    final=CHECKPOINTS/(label+'.zip')
    temporary=CHECKPOINTS/('.'+label+'.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_STORED) as archive:
        for name,value in members.items():archive.writestr(name,value)
    os.replace(temporary,final)
    return {'cloud_archive':str(final),'bytes':final.stat().st_size,'sha256':hashlib.sha256(final.read_bytes()).hexdigest(),
            'ephemeral_runtime_storage':True,'downloaded_to_personal_computer':False}


def checkpoint_pass(suite,label):
    # Serialize the complete multi-shard snapshot against a separate manual
    # driver invocation, not only against this notebook's own launch lock.
    with suite.runner.exclusive_directory(SUITE):
        return checkpoint_pass_locked(suite,label)


def execute_bounded(suite,checkpoint):
    """A finite scheduler; a partial pass is never reported as the final result."""
    final_summary=None
    last=None
    for pass_index in range(1,MAX_PASSES+1):
        last=suite.execute_suite(SUITE,workers=WORKERS,max_rounds=1,max_calls_per_batch=CALLS_PER_SHARD_PER_PASS)
        if last['all_shards_complete']:
            final_summary=suite.score_suite(SUITE)
        receipt=checkpoint(suite,f'{INVOCATION}-pass-{pass_index:02d}')
        print(json.dumps({'pass':pass_index,'maximum_passes':MAX_PASSES,'all_shards_complete':last['all_shards_complete'],
                          'checkpoint':receipt,'result_not_yet_complete':not last['all_shards_complete']}),flush=True)
        if last['all_shards_complete']:break
        unexpected=[r['stop_reason'] for r in last['results'] if r['stop_reason'] not in {'complete','finite_round_cap'}]
        if unexpected:
            print(json.dumps({'execution_stopped':True,'safe_stop_reasons':sorted(set(unexpected)),
                              'automatic_inflight_recovery':False,'final_effectiveness_result':False}),flush=True)
            break
    return {'complete':bool(last and last['all_shards_complete']),'last_execution':last,'final_summary':final_summary}


INVOCATION=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+str(time.time_ns())[-8:]
lock=(WORK/'authorized-launch.lock').open('a')
previous=os.environ.get('CBT_EVAL_API_KEY')
secret=None
try:
    fcntl.flock(lock.fileno(),fcntl.LOCK_EX|fcntl.LOCK_NB)
    suite_driver.runner.require_colab()
    try:
        secret=userdata.get('Capstone')
    except Exception:
        raise RuntimeError('Existing Colab Secret Capstone is unavailable. No provider call started.') from None
    if not isinstance(secret,str) or not secret.strip():raise RuntimeError('Existing Colab Secret is empty. No provider call started.')
    os.environ['CBT_EVAL_API_KEY']=secret
    preparation=suite_driver.prepare_suite(DATA,SUITE,EXPOSURE,mode='full')
    print(json.dumps({'preparation':preparation,'no_fabricated_effectiveness_result':True}),flush=True)
    result=execute_bounded(suite_driver,checkpoint_pass)
    suite_driver._atomic_json(WORK/('launch-receipt-'+INVOCATION+'.json'),result)
    if result['complete']:
        print(json.dumps({'status':'complete','summary':result['final_summary'],'checkpoint_directory':str(CHECKPOINTS)}),flush=True)
    else:
        print(json.dumps({'status':'partial_or_safely_stopped','final_effectiveness_result':False,
                          'checkpoint_directory':str(CHECKPOINTS),'next_action':'Preserve cloud journals/checkpoints; inspect the safe receipt before any further invocation.'}),flush=True)
except BaseException as error:
    print(json.dumps({'status':'safe_launch_error','error_type':type(error).__name__,
                      'raw_error_or_secret_displayed':False,'cloud_artifacts_preserved_at':str(WORK),
                      'final_effectiveness_result':False}),flush=True)
    raise RuntimeError('Cloud launch stopped safely. Inspect retained cloud artifacts; no provider error text or credential is displayed.') from None
finally:
    if previous is None:os.environ.pop('CBT_EVAL_API_KEY',None)
    else:os.environ['CBT_EVAL_API_KEY']=previous
    secret=None
    fcntl.flock(lock.fileno(),fcntl.LOCK_UN)
    lock.close()
